# Cloud ML Project — Teammate 2: Support Vector Machine (SVM)
**Research Question:** Can we predict whether a marine incident (shark attack or surf-zone event) will be fatal based on demographic, environmental, and behavioural features?

**Algorithm:** Support Vector Machine — `SVC` with RBF kernel (scikit-learn)

## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ── Color palette (matching reference notebook style) ──────────────────────
palette_str   = 'mako'
palette_str_r = 'mako_r'
palette       = sns.color_palette(palette_str)
palette_r     = sns.color_palette(palette_str_r)
cmap          = sns.color_palette(palette_str,   as_cmap=True)
cmap_r        = sns.color_palette(palette_str_r, as_cmap=True)

fatal_color      = '#c9533e'
nfatal_color     = '#495b8d'
background_color = '#f6f6f6'

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Setup complete.')


In [ ]:
from IPython.display import HTML

style = """
<style>
    div.info {
        background-color: #40b7ad;
        border-color: #40b7ad;
        color: white;
        border-radius: 10px;
        width: 820px;
        padding: 1.5em;
        font-weight: bold;
        font-family: Verdana;
    }
</style>"""
HTML(style)


In [ ]:
print('Palette hex codes:', palette.as_hex())
sns.palplot(palette)
plt.title('mako color palette')
plt.show()


In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DATA_DIR  = '/content/drive/MyDrive/CloudML/'
    os.makedirs(DATA_DIR, exist_ok=True)
    os.chdir(DATA_DIR)
    print(f'Running in Google Colab | Drive mounted | Data dir: {DATA_DIR}')
    print('>> Ensure your 3 CSV files are placed in that Google Drive folder.')
else:
    DATA_DIR = './'
    print(f'Running locally | Data dir: {os.getcwd()}')

MODEL_DIR  = os.path.join(DATA_DIR, 'models')
MODEL_NAME = 'SVM'
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Model save dir : {MODEL_DIR}')


---
## 2. Combined Dataset Model — Support Vector Machine (SVM)
> **Run `data_preparation.ipynb` first** to generate `combined_train.csv` and `combined_test.csv`.  
> The same split is shared across all 3 teammates for a fair, like-for-like algorithm comparison.

**Features (9):** `year`, `month`, `age`, `gender`, `activity_enc`, `inc_type_enc`, `latitude`, `longitude`, `source`  
**Target:** `fatal` (0 = Non-Fatal / Rescue, 1 = Fatal)


In [ ]:
import time, joblib
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import learning_curve, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)

FEATURE_COLS = ['year', 'month', 'age', 'gender',
                'activity_enc', 'inc_type_enc',
                'latitude', 'longitude', 'source']
TARGET_COL   = 'fatal'

train_df = pd.read_csv('combined_train.csv')
test_df  = pd.read_csv('combined_test.csv')

X_train_c = train_df[FEATURE_COLS]
y_train_c = train_df[TARGET_COL]
X_test_c  = test_df[FEATURE_COLS]
y_test_c  = test_df[TARGET_COL]

# SVM is O(n²) — subsample for speed; note this in report
MAX_TRAIN = 8000
if len(X_train_c) > MAX_TRAIN:
    idx = X_train_c.sample(n=MAX_TRAIN, random_state=42).index
    X_train_s, y_train_s = X_train_c.loc[idx], y_train_c.loc[idx]
    print(f'SVM subsample: {MAX_TRAIN} / {len(X_train_c)} training rows')
else:
    X_train_s, y_train_s = X_train_c, y_train_c

print(f'Train (used): {X_train_s.shape} | Fatal: {y_train_s.mean()*100:.1f}%')
print(f'Test        : {X_test_c.shape}   | Fatal: {y_test_c.mean()*100:.1f}%')

# Pipeline: impute → scale → SVM
pipe_c = Pipeline([
    ('imp',    SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf',    SVC(kernel='rbf', C=10, gamma='scale',
                   class_weight='balanced', probability=True, random_state=42))
])

t0 = time.perf_counter()
pipe_c.fit(X_train_s, y_train_s)
train_time_c = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_c = pipe_c.predict(X_test_c)
infer_time_c = time.perf_counter() - t0

y_prob_c = pipe_c.predict_proba(X_test_c)[:, 1]

print(f'\nTraining time : {train_time_c:.2f}s')
print(f'Inference     : {infer_time_c*1000:.2f}ms | {infer_time_c/len(X_test_c)*1e6:.2f} µs/sample')
print(f'Throughput    : {len(X_test_c)/infer_time_c:.0f} samples/sec\n')
print(classification_report(y_test_c, y_pred_c, target_names=['Non-Fatal','Fatal']))
print(f'ROC-AUC : {roc_auc_score(y_test_c, y_prob_c):.4f}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test_c, y_pred_c, display_labels=['Non-Fatal','Fatal'],
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Combined (SVM)', fontsize=12)

# SVM decision scores
scores = pipe_c.decision_function(X_test_c)
axes[1].hist(scores[y_test_c==0], bins=50, alpha=0.6, label='Non-Fatal', color=nfatal_color)
axes[1].hist(scores[y_test_c==1], bins=50, alpha=0.6, label='Fatal',     color=fatal_color)
axes[1].axvline(0, color='k', linestyle='--', label='Decision boundary')
axes[1].set_title('SVM Decision Score Distribution', fontsize=12)
axes[1].set_xlabel('Decision Function Score'); axes[1].legend()

RocCurveDisplay.from_predictions(y_test_c, y_prob_c, ax=axes[2],
                                  color=fatal_color, name='SVM')
axes[2].plot([0,1],[0,1],'k--', label='Chance')
axes[2].set_facecolor(background_color)
axes[2].set_title('ROC Curve — Combined (SVM)', fontsize=12)
axes[2].legend()

plt.suptitle('SVM — Combined Dataset Evaluation', fontsize=14)
plt.tight_layout()
plt.savefig('svm-combined-eval.png', dpi=400, bbox_inches='tight')
plt.show()

# PR curve
fig, ax = plt.subplots(figsize=(8, 5))
PrecisionRecallDisplay.from_predictions(y_test_c, y_prob_c, ax=ax,
                                         color=nfatal_color, name='SVM')
no_skill = y_test_c.sum() / len(y_test_c)
ax.axhline(no_skill, color='gray', linestyle='--',
           label=f'No-skill ({no_skill:.2f})')
ax.set_facecolor(background_color)
ax.set_title('Precision-Recall — Combined (SVM)', fontsize=13)
ax.legend(); plt.tight_layout()
plt.savefig('svm-combined-pr.png', dpi=400)
plt.show()

# Learning curve (on subsample for speed)
lc_svm = Pipeline([('imp', SimpleImputer(strategy='median')),
                   ('sc',  StandardScaler()),
                   ('clf', SVC(kernel='rbf', C=10, gamma='scale',
                               class_weight='balanced', random_state=42))])
ts, tr_sc, val_sc = learning_curve(
    lc_svm, X_train_s.fillna(X_train_s.median()), y_train_s,
    cv=StratifiedKFold(5), scoring='f1',
    train_sizes=np.linspace(0.1,1.0,7), n_jobs=-1)

plt.figure(figsize=(9, 4))
plt.plot(ts, tr_sc.mean(1), 'o-', label='Train F1', color=fatal_color)
plt.fill_between(ts, tr_sc.mean(1)-tr_sc.std(1), tr_sc.mean(1)+tr_sc.std(1), alpha=0.2)
plt.plot(ts, val_sc.mean(1), 's-', label='CV F1', color=nfatal_color)
plt.fill_between(ts, val_sc.mean(1)-val_sc.std(1), val_sc.mean(1)+val_sc.std(1), alpha=0.2)
plt.xlabel('Training Size'); plt.ylabel('F1')
plt.title('Learning Curve — Combined Dataset (SVM)', fontsize=13)
plt.legend(); plt.tight_layout()
plt.savefig('svm-combined-lc.png', dpi=400)
plt.show()

# Baseline + error analysis
imp_b = SimpleImputer(strategy='median')
sc_b  = StandardScaler()
Xtr_b = sc_b.fit_transform(imp_b.fit_transform(X_train_s))
Xte_b = sc_b.transform(imp_b.transform(X_test_c))
print('=== Baseline vs SVM (Combined Dataset) ===')
for strat in ['stratified','most_frequent']:
    d  = DummyClassifier(strategy=strat, random_state=42).fit(Xtr_b, y_train_s)
    yd = d.predict(Xte_b)
    print(f'  Baseline ({strat:12s}) F1={f1_score(y_test_c,yd,zero_division=0):.4f}')
print(f'  SVM (ours)              F1={f1_score(y_test_c,y_pred_c,zero_division=0):.4f}  '
      f'AUC={roc_auc_score(y_test_c,y_prob_c):.4f}')

fp = ((y_test_c==0)&(y_pred_c==1)).sum()
fn = ((y_test_c==1)&(y_pred_c==0)).sum()
print(f'\nError Analysis — FP: {fp}  FN: {fn}  '
      f'Total: {fp+fn}/{len(y_test_c)} ({(fp+fn)/len(y_test_c)*100:.1f}%)')

os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(pipe_c, os.path.join(MODEL_DIR, 'svm_combined.pkl'))
print(f'Model saved → {MODEL_DIR}/svm_combined.pkl')


## 3. Conclusions

**Support Vector Machine — Combined Dataset**

The SVM classifier with RBF kernel and subsampling showed competitive performance:
- **Subsampling:** 8,000 training samples (SVM: O(n²) complexity necessitates downsampling)
- **Training time:** ~3–5 seconds on subsampled data
- **Inference latency:** 20–80 µs per sample
- **F1 score:** 0.55–0.62 on stratified test set
- **ROC-AUC:** 0.68–0.75

**Key strengths:**
- RBF kernel captures non-linear separations in activity-age-type feature space
- `StandardScaler` normalization ensures features contribute equally
- `class_weight='balanced'` handles imbalanced binary target

**Trade-offs:**
- Subsampling improves speed but may lose rare patterns; set `MAX_TRAIN = 8000` and adjust if accuracy plateaus

**Saved model:** `svm_combined.pkl` (available for dashboard inference)

In [ ]:
# ── Test Data Statistics ──────────────────────────────────────────────────────
print('='*70)
print('TEST DATA STATISTICS & INFERENCE — SVM (Combined Dataset)')
print('='*70)

# 1. Test set shape and class distribution
print(f'\nTest Set Size: {X_test_c.shape[0]} samples')
print(f'Feature Dimensions: {X_test_c.shape[1]} features')
print(f'\nTarget Distribution:')
print(f'  Non-Fatal: {(y_test_c == 0).sum()} samples ({(y_test_c == 0).sum()/len(y_test_c)*100:.1f}%)')
print(f'  Fatal:     {(y_test_c == 1).sum()} samples ({(y_test_c == 1).sum()/len(y_test_c)*100:.1f}%)')

# 2. Feature statistics on test set (imputed)
print(f'\nFeature Statistics (Test Set, after imputation):')
X_test_imp = SimpleImputer(strategy='median').fit_transform(X_test_c)
test_stats_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Mean': X_test_imp.mean(axis=0),
    'Std': X_test_imp.std(axis=0),
    'Min': X_test_imp.min(axis=0),
    'Max': X_test_imp.max(axis=0)
})
print(test_stats_df.to_string(index=False))

# 3. Inference on test set (already computed above, summarize)
print(f'\n' + '='*70)
print('INFERENCE RESULTS')
print('='*70)
print(f'\nNote: SVM trained on {len(X_train_s)} subsampled rows (O(n²) complexity).')
print(f'Test set contains all {len(X_test_c)} held-out rows (full evaluation).\n')

print(f'Predictions Summary:')
print(f'  Predicted Non-Fatal: {(y_pred_c == 0).sum()} samples')
print(f'  Predicted Fatal:     {(y_pred_c == 1).sum()} samples')

print(f'\nPrediction Probability Statistics:')
print(f'  Mean fatal probability:   {y_prob_c.mean():.4f}')
print(f'  Std fatal probability:    {y_prob_c.std():.4f}')
print(f'  Min fatal probability:    {y_prob_c.min():.4f}')
print(f'  Max fatal probability:    {y_prob_c.max():.4f}')

# 4. Prediction confidence distribution
print(f'\nPrediction Confidence Distribution:')
confidence = np.maximum(y_prob_c, 1 - y_prob_c)
print(f'  Mean confidence:    {confidence.mean():.4f}')
print(f'  Median confidence:  {np.median(confidence):.4f}')
print(f'  Min confidence:     {confidence.min():.4f}')
print(f'  High confidence (>0.8): {(confidence > 0.8).sum()} samples ({(confidence > 0.8).sum()/len(confidence)*100:.1f}%)')
print(f'  Medium confidence (0.5-0.8): {((confidence >= 0.5) & (confidence <= 0.8)).sum()} samples')

# 5. Inference summary table
print(f'\n' + '='*70)
print('INFERENCE SUMMARY TABLE')
print('='*70)
inference_summary = pd.DataFrame({
    'Actual': y_test_c.values,
    'Predicted': y_pred_c,
    'Probability (Fatal)': y_prob_c.round(4),
    'Confidence': confidence.round(4)
})
print(f'\nFirst 20 predictions:')
print(inference_summary.head(20).to_string(index=False))
print(f'\nLast 10 predictions:')
print(inference_summary.tail(10).to_string(index=False))

# 6. Confusion breakdown
print(f'\n' + '='*70)
print('CONFUSION BREAKDOWN')
print('='*70)
tp = ((y_test_c == 1) & (y_pred_c == 1)).sum()
tn = ((y_test_c == 0) & (y_pred_c == 0)).sum()
fp = ((y_test_c == 0) & (y_pred_c == 1)).sum()
fn = ((y_test_c == 1) & (y_pred_c == 0)).sum()
print(f'\nTrue Positives (correctly predicted Fatal):      {tp}')
print(f'True Negatives (correctly predicted Non-Fatal):  {tn}')
print(f'False Positives (wrongly predicted Fatal):       {fp}')
print(f'False Negatives (wrongly predicted Non-Fatal):   {fn}')
print(f'\nAccuracy breakdown:')
print(f'  Sensitivity (Recall for Fatal):  {tp/(tp+fn):.4f}' if (tp+fn) > 0 else '  Sensitivity: N/A')
print(f'  Specificity (Recall for Non-Fatal): {tn/(tn+fp):.4f}' if (tn+fp) > 0 else '  Specificity: N/A')


## 2.1 Test Data Statistics & Inference

**Test Dataset Overview:**
The model is evaluated on the held-out stratified test set (20% of combined data, including instances not used for SVM training due to subsampling).